# Análisis histórico de los Juegos Olímpicos con Apache Spark

**Magíster en Ciencia de Datos e Inteligencia Artificial · MCDI502 · Taller I**  
**Grupo 2**  
**Integrantes:** Verónica Durán Cisterna, Raúl Moya Arriagada, Daniela Rojas Vilches y Manuel Sánchez Cárcamo  
**Fecha:** 8 de noviembre de 2026

Este trabajo desarrolla un análisis reproducible de datos históricos olímpicos utilizando RDD, DataFrames y Spark SQL. Se examinan la participación de deportistas, la distribución de medallas y estadísticas descriptivas de variables antropométricas. Los resultados dependen de la cobertura y calidad de los archivos de origen.


## 1. Entorno de procesamiento

El estudio se ejecuta en Visual Studio Code mediante Python 3.13, Java 17 y Apache Spark 4.0.5. Se configura el intérprete local antes de iniciar Spark para habilitar tanto las operaciones con DataFrames como las transformaciones distribuidas con RDD.

In [10]:
import csv
import os
import sys
from pathlib import Path
from pyspark.sql import SparkSession, functions as F, types as T

# Los procesos Python de Spark utilizan el mismo intérprete que Jupyter.
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

spark = (SparkSession.builder.master('local[2]')
         .appName('MCDI502_Analisis_Olimpico_Grupo2')
         .config('spark.pyspark.python', sys.executable)
         .config('spark.pyspark.driver.python', sys.executable)
         .config('spark.python.worker.reuse', 'false')
         .getOrCreate())
sc = spark.sparkContext
print('Python:', sys.version.split()[0], '| Spark:', spark.version)


Python: 3.13.17 | Spark: 4.0.5


### 1.1. Fuentes de información

Los archivos de origen se mantienen sin modificaciones. Se comprueba su disponibilidad y se utilizan rutas relativas a la carpeta del proyecto.

In [11]:
base = Path.cwd()
NECESARIOS = ['deportista2.csv', 'equipo.csv', 'evento.csv', 'resultados.csv', 'juegos.json']
rutas = {nombre: str(base / nombre) for nombre in NECESARIOS}
faltantes = [nombre for nombre in NECESARIOS if not (base / nombre).is_file()]
if faltantes:
    raise FileNotFoundError(f'Faltan archivos en {base}: {", ".join(faltantes)}')
print('Fuentes disponibles:', ', '.join(NECESARIOS))

def separador_csv(nombre):
    with open(rutas[nombre], encoding='utf-8-sig', newline='') as archivo:
        muestra = archivo.read(32768)
    try:
        return csv.Sniffer().sniff(muestra, delimiters=',;|\t').delimiter
    except csv.Error:
        primera = muestra.splitlines()[0]
        return max([',', ';', '|', '\t'], key=primera.count)


Fuentes disponibles: deportista2.csv, equipo.csv, evento.csv, resultados.csv, juegos.json


## 2. Procesamiento distribuido de deportistas

Se construyen dos subconjuntos complementarios a partir de los identificadores de posición del archivo disponible. Su unión conserva la totalidad de los registros y se redistribuye en seis particiones.

In [12]:
rdd_archivo = sc.textFile(rutas['deportista2.csv'], minPartitions=6)
separador_deportista = separador_csv('deportista2.csv')

def parsear_deportista(linea):
    fila = next(csv.reader([linea], delimiter=separador_deportista))
    if len(fila) != 7:
        raise ValueError(f'Registro de deportista con {len(fila)} campos: {fila[:3]}')
    return fila

def entero(x):
    try:
        return int(str(x).strip()) if x is not None and str(x).strip().upper() not in ('', 'NA', 'NULL') else None
    except (ValueError, TypeError):
        return None

def decimal(x):
    try:
        return float(str(x).strip()) if x is not None and str(x).strip().upper() not in ('', 'NA', 'NULL') else None
    except (ValueError, TypeError):
        return None

rdd_parseado = rdd_archivo.map(parsear_deportista)
rdd_indice = rdd_parseado.zipWithIndex()
rdd_deportista = rdd_indice.filter(lambda par: par[1] % 2 == 0).keys()
rdd_deportista2 = rdd_indice.filter(lambda par: par[1] % 2 != 0).keys()
deportistaTotal = rdd_deportista.union(rdd_deportista2).repartition(6)
conteo_origen = rdd_parseado.count()
conteo_union = deportistaTotal.count()
assert conteo_origen == conteo_union, 'La unión no conservó el número de registros.'
assert deportistaTotal.getNumPartitions() == 6
print('Registros de origen:', conteo_origen, '| Registros unidos:', conteo_union)
print('Particiones del RDD:', deportistaTotal.getNumPartitions())


Registros de origen: 67785 | Registros unidos: 67785
Particiones del RDD: 6


In [13]:
esquema_deportista = T.StructType([
    T.StructField('deportista_id', T.IntegerType(), True),
    T.StructField('nombre', T.StringType(), True),
    T.StructField('genero', T.IntegerType(), True),
    T.StructField('edad', T.IntegerType(), True),
    T.StructField('altura', T.DoubleType(), True),
    T.StructField('peso', T.DoubleType(), True),
    T.StructField('equipo_id', T.IntegerType(), True)
])
rdd_tipos = deportistaTotal.map(lambda x: (
    entero(x[0]), x[1], entero(x[2]), entero(x[3]),
    decimal(x[4]), decimal(x[5]), entero(x[6])
))
deportista = spark.createDataFrame(rdd_tipos, esquema_deportista)
deportista.printSchema()
deportista.show(5, truncate=False)


root
 |-- deportista_id: integer (nullable = true)
 |-- nombre: string (nullable = true)
 |-- genero: integer (nullable = true)
 |-- edad: integer (nullable = true)
 |-- altura: double (nullable = true)
 |-- peso: double (nullable = true)
 |-- equipo_id: integer (nullable = true)

+-------------+---------------+------+----+------+----+---------+
|deportista_id|nombre         |genero|edad|altura|peso|equipo_id|
+-------------+---------------+------+----+------+----+---------+
|67847        |Lee DuHaeng    |1     |30  |180.0 |64.0|970      |
|67849        |Edward Lee     |1     |0   |0.0   |0.0 |199      |
|67851        |Eric George Lee|1     |25  |0.0   |0.0 |413      |
|67853        |Lee EunByeol   |2     |18  |154.0 |46.0|970      |
|67855        |Lee EunGyeong  |2     |20  |0.0   |0.0 |970      |
+-------------+---------------+------+----+------+----+---------+
only showing top 5 rows


### 2.1. Transformaciones sobre deportistas

Se filtran los registros de personas mayores de edad y de mujeres, y se aplica una transformación de normalización de texto a mayúsculas.

In [14]:
MayorEdad = deportistaTotal.filter(lambda x: entero(x[3]) is not None and entero(x[3]) >= 18)
Deportistas_mujer = deportistaTotal.filter(lambda x: entero(x[2]) == 2)
Deportistas_mayusculas = deportistaTotal.map(
    lambda x: [campo.upper() if isinstance(campo, str) else campo for campo in x]
)
print('Mayores de edad:', MayorEdad.count())
print('Mujeres:', Deportistas_mujer.count())
print('Ejemplos normalizados:', Deportistas_mayusculas.take(2))


Mayores de edad: 61679
Mujeres: 17575
Ejemplos normalizados: [['67847', 'LEE DUHAENG', '1', '30', '180', '64', '970'], ['67849', 'EDWARD LEE', '1', '0', '0', '0', '199']]


## 3. Preparación de DataFrames

Se interpretan los separadores y las comillas de cada fuente antes de normalizar las claves. El catálogo de eventos se lee mediante el módulo estándar `csv` para resolver de forma explícita los registros que contienen comas dentro de nombres deportivos, incluso cuando el archivo incluye comillas escapadas.

In [15]:

import csv
from pyspark.sql import types as T

def leer_csv_spark(nombre):
    return (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("sep", separador_csv(nombre))
        .option("quote", '"')
        .option("escape", "\\")
        .csv(rutas[nombre])
    )


def interpretar_fila_evento(texto):
    """Interpreta registros CSV con nombres de eventos entrecomillados."""

    candidatos = [texto.strip()]

    if texto.startswith('"') and texto.endswith('"'):
        candidatos.append(texto[1:-1])

    candidatos.extend([
        valor.replace('\\"', '"')
        for valor in list(candidatos)
    ])

    for candidato in candidatos:
        try:
            campos = next(
                csv.reader(
                    [candidato],
                    delimiter=",",
                    quotechar='"',
                    escapechar="\\"
                )
            )

            if len(campos) == 1:
                campos = next(
                    csv.reader(
                        [campos[0]],
                        delimiter=",",
                        quotechar='"',
                        escapechar="\\"
                    )
                )

            if len(campos) == 3:
                return [valor.strip() for valor in campos]

        except (csv.Error, StopIteration):
            continue

    raise ValueError(
        f"Fila de evento no interpretable: {texto[:180]!r}"
    )


def convertir_entero(valor):
    """Convierte identificadores numéricos y conserva valores faltantes."""

    if valor is None:
        return None

    valor = str(valor).strip()

    if valor.upper() in ("", "#N/A", "NA", "NULL", "NONE"):
        return None

    return int(valor)


def cargar_eventos(ruta):
    """Carga los eventos deportivos conservando los valores faltantes."""

    with open(ruta, encoding="utf-8-sig", newline="") as archivo:
        lineas = [
            linea.rstrip("\r\n")
            for linea in archivo
            if linea.strip()
        ]

    if not lineas:
        raise ValueError("evento.csv está vacío.")

    cabecera = interpretar_fila_evento(lineas[0])

    esperadas = ["cod_evento", "evento", "deporte_id"]

    if [columna.lower() for columna in cabecera] != esperadas:
        raise ValueError(
            f"Encabezado no esperado: {cabecera}. "
            f"Se requiere: {esperadas}"
        )

    registros = [
        interpretar_fila_evento(linea)
        for linea in lineas[1:]
    ]

    registros_convertidos = [
        (
            convertir_entero(fila[0]),
            fila[1],
            convertir_entero(fila[2])
        )
        for fila in registros
    ]

    # La clave principal debe ser válida.
    # deporte_id puede ser NULL cuando el origen contiene #N/A.
    incorrectos = [
        (indice + 2, fila)
        for indice, fila in enumerate(registros_convertidos)
        if fila[0] is None
    ]

    if incorrectos:
        raise ValueError(
            f"Identificadores de evento inválidos: {incorrectos[:3]}"
        )

    esquema = T.StructType([
        T.StructField("evento_id", T.IntegerType(), False),
        T.StructField("evento", T.StringType(), True),
        T.StructField("deporte_id", T.IntegerType(), True)
    ])

    return spark.createDataFrame(
        registros_convertidos,
        schema=esquema
    )


# Carga de los conjuntos de datos

Equipo = leer_csv_spark("equipo.csv")

Evento = cargar_eventos(rutas["evento.csv"])

Resultado = leer_csv_spark("resultados.csv")

Juego = (
    spark.read
    .option("multiLine", True)
    .json(rutas["juegos.json"])
)


# Resumen de los conjuntos de datos

for nombre, df in [
    ("Equipo", Equipo),
    ("Evento", Evento),
    ("Resultado", Resultado),
    ("Juego", Juego)
]:
    print(
        nombre,
        "| Registros:", df.count(),
        "| Columnas:", df.columns
    )


# Verificación de valores faltantes en eventos

print(
    "Eventos sin deporte asociado:",
    Evento.filter(Evento.deporte_id.isNull()).count()
)


Equipo | Registros: 1184 | Columnas: ['id', 'equipo', 'sigla']
Evento | Registros: 765 | Columnas: ['evento_id', 'evento', 'deporte_id']
Resultado | Registros: 271116 | Columnas: ['resultado_id', 'medalla', 'deportista_id', 'juego_id', 'evento_id']
Juego | Registros: 51 | Columnas: ['ano', 'ciudad', 'juego_id', 'temporada']
Eventos sin deporte asociado: 2


### 3.1. Estandarización y control de claves

Se verifica la estructura de las fuentes y se comprueba la unicidad de las claves de las tablas descriptivas antes de integrar los registros.

In [17]:

from pyspark.sql import functions as F

# Validación de columnas requeridas
def exigir_columnas(df, nombre, columnas):
    faltan = sorted(set(columnas) - set(df.columns))

    if faltan:
        raise ValueError(
            f"{nombre}: faltan columnas {faltan}. "
            f"Disponibles: {df.columns}"
        )


# Conversión y validación de claves numéricas
def clave_entera(df, nombre, columna):

    valores = F.trim(F.col(columna).cast("string"))

    faltantes = ["", "NA", "N/A", "#N/A", "NULL", "NONE"]

    valor_normalizado = F.upper(valores)

    invalidos = df.filter(
        valores.isNotNull()
        & ~valor_normalizado.isin(*faltantes)
        & ~valores.rlike(r"^-?[0-9]+$")
    ).select(columna).limit(3).collect()

    if invalidos:
        raise ValueError(
            f"{nombre}.{columna}: valores no numéricos: {invalidos}"
        )

    return df.withColumn(
        columna,
        F.when(
            valores.isNull() | valor_normalizado.isin(*faltantes),
            F.lit(None).cast("int")
        ).otherwise(valores.cast("int"))
    )


# Verificación de estructuras
if "id" in Equipo.columns:
    Equipo = Equipo.withColumnRenamed("id", "equipo_id")

exigir_columnas(
    Equipo, "Equipo",
    ["equipo_id", "equipo", "sigla"]
)

exigir_columnas(
    Evento, "Evento",
    ["evento_id", "evento", "deporte_id"]
)

exigir_columnas(
    Resultado, "Resultado",
    ["deportista_id", "evento_id", "juego_id", "medalla"]
)

exigir_columnas(
    Juego, "Juego",
    ["juego_id", "temporada", "ciudad"]
)


# Normalización de claves
Equipo = clave_entera(
    Equipo, "Equipo", "equipo_id"
)

for columna in ["deportista_id", "evento_id", "juego_id"]:
    Resultado = clave_entera(
        Resultado, "Resultado", columna
    )

Juego = clave_entera(
    Juego, "Juego", "juego_id"
)


# Comprobación de integridad de identificadores
tablas = [
    ("Deportista", deportista, "deportista_id"),
    ("Equipo", Equipo, "equipo_id"),
    ("Evento", Evento, "evento_id"),
    ("Juego", Juego, "juego_id")
]

for nombre, df, clave in tablas:

    duplicados = (
        df.groupBy(clave)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    nulos = (
        df.filter(F.col(clave).isNull())
        .count()
    )

    print(
        f"{nombre}: "
        f"claves duplicadas = {duplicados}; "
        f"claves nulas = {nulos}"
    )

    if duplicados > 0 or nulos > 0:
        raise ValueError(
            f"La clave {clave} de {nombre} "
            f"requiere revisión."
        )

print("Validación de claves completada correctamente.")



Deportista: claves duplicadas = 0; claves nulas = 0
Equipo: claves duplicadas = 0; claves nulas = 0
Evento: claves duplicadas = 0; claves nulas = 0
Juego: claves duplicadas = 0; claves nulas = 0
Validación de claves completada correctamente.


### 3.2. Optimización de dimensiones

Se mantienen en caché las tablas descriptivas de menor tamaño para reutilizarlas durante las consultas. En la integración se emplea difusión de las dimensiones pequeñas.

In [18]:
Equipo = Equipo.cache()
Evento = Evento.cache()
Juego = Juego.cache()
for dimension in (Equipo, Evento, Juego):
    dimension.count()
print('Dimensiones en caché:', 'Equipo, Evento, Juego')


Dimensiones en caché: Equipo, Evento, Juego


## 4. Integración y cobertura

La tabla de resultados constituye la base del análisis. Las uniones izquierdas preservan las observaciones originales, mientras que los controles de cobertura identifican las relaciones que no encuentran correspondencia.

In [19]:
sin_deportista = Resultado.join(deportista.select('deportista_id'), 'deportista_id', 'left_anti').count()
print('Resultados sin deportista asociado:', sin_deportista)

Integrado = (Resultado
    .join(deportista, 'deportista_id', 'left')
    .join(F.broadcast(Equipo), 'equipo_id', 'left')
    .join(F.broadcast(Evento), 'evento_id', 'left')
    .join(F.broadcast(Juego), 'juego_id', 'left'))

# Se interpretan los campos del JSON de acuerdo con los valores observados en esta fuente.
Integrado = (Integrado
    .withColumn('anio_juego', F.col('temporada').cast('int'))
    .withColumn('temporada_juego', F.col('ciudad')))

conteo_resultados = Resultado.count()
conteo_integrado = Integrado.count()
print('Resultados originales:', conteo_resultados, '| Integrados:', conteo_integrado)
assert conteo_integrado == conteo_resultados, 'Las uniones alteraron la cantidad de resultados.'
Integrado.select('deportista_id', 'nombre', 'equipo', 'evento', 'medalla',
                 'anio_juego', 'temporada_juego').show(10, truncate=False)


Resultados sin deportista asociado: 134720
Resultados originales: 271116 | Integrados: 271116
+-------------+----------------+-------+----------------------------------------+-------+----------+---------------+
|deportista_id|nombre          |equipo |evento                                  |medalla|anio_juego|temporada_juego|
+-------------+----------------+-------+----------------------------------------+-------+----------+---------------+
|1            |NULL            |NULL   |Basketball Men's Basketball             |NA     |1992      |Verano         |
|3            |NULL            |NULL   |Football Men's Football                 |NA     |1920      |Verano         |
|5            |NULL            |NULL   |Speed Skating Women's 500 metres        |NA     |1988      |Invierno       |
|5            |NULL            |NULL   |Speed Skating Women's 1000 metres       |NA     |1988      |Invierno       |
|5            |NULL            |NULL   |Speed Skating Women's 500 metres        |NA    

## 5. Particionamiento e inspección

El conjunto integrado se redistribuye en cinco particiones para establecer una organización explícita de las tareas de procesamiento.

In [20]:
Integrado = Integrado.repartition(5)
print('Particiones del DataFrame:', Integrado.rdd.getNumPartitions())
print('Registros:', Integrado.count())
Integrado.printSchema()
Integrado.show(5, truncate=False)


Particiones del DataFrame: 5
Registros: 271116
root
 |-- juego_id: integer (nullable = true)
 |-- evento_id: integer (nullable = true)
 |-- equipo_id: integer (nullable = true)
 |-- deportista_id: integer (nullable = true)
 |-- resultado_id: string (nullable = true)
 |-- medalla: string (nullable = true)
 |-- nombre: string (nullable = true)
 |-- genero: integer (nullable = true)
 |-- edad: integer (nullable = true)
 |-- altura: double (nullable = true)
 |-- peso: double (nullable = true)
 |-- equipo: string (nullable = true)
 |-- sigla: string (nullable = true)
 |-- evento: string (nullable = true)
 |-- deporte_id: integer (nullable = true)
 |-- ano: string (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- temporada: long (nullable = true)
 |-- anio_juego: integer (nullable = true)
 |-- temporada_juego: string (nullable = true)

+--------+---------+---------+-------------+------------+-------+--------------------------+------+----+------+----+-------+-----+-----------------

## 6. Variables derivadas

Se calcula el índice de masa corporal (IMC) a partir del peso en kilogramos y la altura en centímetros, cuando ambas medidas son válidas. El sexo se presenta mediante etiquetas descriptivas.

In [21]:
Integrado = (Integrado
    .withColumn('IMC', F.when((F.col('altura') > 0) & (F.col('peso') > 0),
                              F.round(F.col('peso') / F.pow(F.col('altura') / 100.0, 2), 2)))
    .withColumn('Descripción_sexo',
                F.when(F.col('genero') == 1, 'Hombre')
                 .when(F.col('genero') == 2, 'Mujer')
                 .otherwise('No informado')))
Integrado.select('nombre', 'altura', 'peso', 'IMC', 'Descripción_sexo').show(10, truncate=False)


+----------------------------------------+------+----+-----+----------------+
|nombre                                  |altura|peso|IMC  |Descripción_sexo|
+----------------------------------------+------+----+-----+----------------+
|Javier Noriega Sanz                     |181.0 |71.0|21.67|Hombre          |
|Tina Pisnik                             |172.0 |65.0|21.97|Mujer           |
|Sidney Edward Ramsden                   |183.0 |0.0 |NULL |Hombre          |
|Moustafa Sadek                          |0.0   |0.0 |NULL |Hombre          |
|Ulrich Peters                           |178.0 |71.0|22.41|Hombre          |
|Frdric Pierre                           |172.0 |66.0|22.31|Hombre          |
|Gsta Oskar Leonard Sandberg             |0.0   |0.0 |NULL |Hombre          |
|Ghislain Gabriel Perrier                |179.0 |75.0|23.41|Hombre          |
|Diego F. Minguens                       |172.0 |68.0|22.99|Hombre          |
|Galina Yakovlevna Minaicheva Sharabidze |0.0   |0.0 |NULL |Muje

## 7. Análisis descriptivo

Las medallas se cuentan por registro de resultado, por lo que una misma medalla de equipo puede corresponder a varios deportistas. Los indicadores de edad y altura consideran únicamente los valores disponibles.

### 7.1. Medallas por equipo


In [22]:
# Medallas por equipo
medallas_equipo = (Integrado.filter(F.col('medalla').isin('Gold','Silver','Bronze'))
    .groupBy('equipo')
    .agg(F.sum(F.when(F.col('medalla')=='Gold',1).otherwise(0)).alias('oro'),
         F.sum(F.when(F.col('medalla')=='Silver',1).otherwise(0)).alias('plata'),
         F.sum(F.when(F.col('medalla')=='Bronze',1).otherwise(0)).alias('bronce'))
    .orderBy(F.desc('oro'),F.desc('plata'),F.desc('bronce')))
medallas_equipo.show(20,truncate=False)


+-------------+----+-----+------+
|equipo       |oro |plata|bronce|
+-------------+----+-----+------+
|NULL         |6571|6475 |6527  |
|United States|1224|729  |604   |
|Soviet Union |608 |414  |372   |
|Germany      |322 |285  |314   |
|Italy        |280 |261  |235   |
|Great Britain|259 |274  |282   |
|Sweden       |241 |230  |234   |
|China        |237 |236  |194   |
|East Germany |218 |188  |132   |
|Canada       |199 |203  |190   |
|Hungary      |181 |153  |178   |
|Netherlands  |174 |179  |218   |
|France       |173 |217  |245   |
|Russia       |166 |156  |181   |
|Australia    |165 |223  |253   |
|Japan        |155 |176  |221   |
|Norway       |127 |162  |125   |
|Finland      |124 |159  |265   |
|Romania      |91  |106  |155   |
|South Korea  |91  |92   |70    |
+-------------+----+-----+------+
only showing top 20 rows


### 7.2. Edad por tipo de medalla


In [23]:
# Edad por tipo de medalla
edad_medalla = (Integrado.filter(F.col('medalla').isin('Gold','Silver','Bronze'))
    .groupBy('medalla')
    .agg(F.count('edad').alias('observaciones_con_edad'),
         F.sum('edad').alias('suma_edad'),
         F.round(F.avg('edad'),2).alias('promedio_edad'),
         F.max('edad').alias('maximo_edad'),
         F.min('edad').alias('minimo_edad')))
edad_medalla.show(truncate=False)


+-------+----------------------+---------+-------------+-----------+-----------+
|medalla|observaciones_con_edad|suma_edad|promedio_edad|maximo_edad|minimo_edad|
+-------+----------------------+---------+-------------+-----------+-----------+
|Silver |6641                  |154548   |23.27        |71         |0          |
|Gold   |6801                  |159787   |23.49        |63         |0          |
|Bronze |6768                  |156627   |23.14        |71         |0          |
+-------+----------------------+---------+-------------+-----------+-----------+



### 7.3. Altura por temporada


In [24]:
# Altura por temporada
temporada = (Integrado.groupBy('temporada_juego')
    .agg(F.count('altura').alias('observaciones_con_altura'),
         F.sum('altura').alias('suma_altura'),
         F.round(F.avg('altura'),2).alias('promedio_altura'),
         F.max('altura').alias('maximo_altura'),
         F.min('altura').alias('minimo_altura'))
    .orderBy('temporada_juego'))
temporada.show(truncate=False)


+---------------+------------------------+-----------+---------------+-------------+-------------+
|temporada_juego|observaciones_con_altura|suma_altura|promedio_altura|maximo_altura|minimo_altura|
+---------------+------------------------+-----------+---------------+-------------+-------------+
|Invierno       |24291                   |3546922.0  |146.02         |203.0        |0.0          |
|Verano         |112105                  |1.5152543E7|135.16         |226.0        |0.0          |
+---------------+------------------------+-----------+---------------+-------------+-------------+



### 7.4. Edad por sexo


In [25]:
# Edad por sexo
sexo = (Integrado.groupBy('Descripción_sexo')
    .agg(F.count('edad').alias('observaciones_con_edad'),
         F.sum('edad').alias('suma_edad'),
         F.round(F.avg('edad'),2).alias('promedio_edad'),
         F.max('edad').alias('maximo_edad'),
         F.min('edad').alias('minimo_edad'))
    .orderBy('Descripción_sexo'))
sexo.show(truncate=False)


+----------------+----------------------+---------+-------------+-----------+-----------+
|Descripción_sexo|observaciones_con_edad|suma_edad|promedio_edad|maximo_edad|minimo_edad|
+----------------+----------------------+---------+-------------+-----------+-----------+
|Hombre          |97568                 |2272281  |23.29        |97         |0          |
|Mujer           |38828                 |842239   |21.69        |74         |0          |
|No informado    |0                     |NULL     |NULL         |NULL       |NULL       |
+----------------+----------------------+---------+-------------+-----------+-----------+



## 8. Consultas con Spark SQL

Se utiliza una vista temporal para contrastar los resultados de agregación por equipo con una consulta SQL equivalente.

In [26]:
Integrado.createOrReplaceTempView('olimpicos')
consulta_sql = spark.sql("""
SELECT equipo,
       SUM(CASE WHEN medalla = 'Gold' THEN 1 ELSE 0 END) AS oro,
       SUM(CASE WHEN medalla = 'Silver' THEN 1 ELSE 0 END) AS plata,
       SUM(CASE WHEN medalla = 'Bronze' THEN 1 ELSE 0 END) AS bronce
FROM olimpicos
WHERE medalla IN ('Gold','Silver','Bronze') AND equipo IS NOT NULL
GROUP BY equipo
ORDER BY oro DESC, plata DESC, bronce DESC
""")
consulta_sql.show(15,truncate=False)


+-------------+----+-----+------+
|equipo       |oro |plata|bronce|
+-------------+----+-----+------+
|United States|1224|729  |604   |
|Soviet Union |608 |414  |372   |
|Germany      |322 |285  |314   |
|Italy        |280 |261  |235   |
|Great Britain|259 |274  |282   |
|Sweden       |241 |230  |234   |
|China        |237 |236  |194   |
|East Germany |218 |188  |132   |
|Canada       |199 |203  |190   |
|Hungary      |181 |153  |178   |
|Netherlands  |174 |179  |218   |
|France       |173 |217  |245   |
|Russia       |166 |156  |181   |
|Australia    |165 |223  |253   |
|Japan        |155 |176  |221   |
+-------------+----+-----+------+
only showing top 15 rows


## 9. Calidad y cobertura de la información

Se cuantifica la cobertura de variables y se informa la distribución de medallas. Las estadísticas reflejan las observaciones disponibles, sin imputación de valores faltantes.

In [27]:
resumen_cobertura = Integrado.agg(
    F.count('*').alias('resultados_totales'),
    F.count('nombre').alias('resultados_con_deportista'),
    F.count('edad').alias('resultados_con_edad'),
    F.count('altura').alias('resultados_con_altura'),
    F.count('equipo').alias('resultados_con_equipo'))
resumen_cobertura.show(truncate=False)
print('Particiones finales:', Integrado.rdd.getNumPartitions())
Integrado.groupBy('medalla').count().orderBy('medalla').show(truncate=False)


+------------------+-------------------------+-------------------+---------------------+---------------------+
|resultados_totales|resultados_con_deportista|resultados_con_edad|resultados_con_altura|resultados_con_equipo|
+------------------+-------------------------+-------------------+---------------------+---------------------+
|271116            |136396                   |136396             |136396               |136290               |
+------------------+-------------------------+-------------------+---------------------+---------------------+

Particiones finales: 5
+-------+------+
|medalla|count |
+-------+------+
|Bronze |13295 |
|Gold   |13372 |
|NA     |231333|
|Silver |13116 |
+-------+------+



## 10. Conclusiones metodológicas

El análisis integra técnicas de procesamiento distribuido con RDD, DataFrames y Spark SQL. Se verifican la conservación de registros y la unicidad de las claves descriptivas, y se presentan indicadores de participación y características de deportistas. La interpretación de los resultados considera la cobertura de los archivos de origen y la contabilización de medallas por registro individual.

La reproducibilidad requiere ejecutar todas las celdas con los archivos originales disponibles en la carpeta del proyecto.